# 12 - Multi-Factor Signal

Combine every factor into one vote.


## Setup

In [1]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Fetch everything

In [2]:
eth = fetch("ETHUSDT", "1d", 500)
from analysis.indicators import add_all
d = add_all(eth).dropna()
last = d.iloc[-1]
print("price:", last.close)

price: 2687.23


## 2. Vote tally

In [3]:
votes = {}

# 1. trend
votes["trend"] = 1 if last.close > last.sma_50 else -1
# 2. long-term trend
votes["trend_lt"] = 1 if last.sma_50 > last.sma_200 else -1
# 3. RSI
votes["rsi"] = 1 if last.rsi_14 < 30 else (-1 if last.rsi_14 > 70 else 0)
# 4. MACD
votes["macd"] = 1 if last.macd_hist > 0 else -1
# 5. drawdown
dd = (last.close / d.close.cummax().iloc[-1] - 1)
votes["drawdown"] = 1 if dd < -0.3 else 0
# 6. funding
try:
    fr = sql("SELECT rate FROM funding_rate ORDER BY ts DESC LIMIT 1")["rate"].iloc[0]
    votes["funding"] = -1 if fr > 0.001 else (1 if fr < -0.001 else 0)
except: votes["funding"] = 0
# 7. F&G
try:
    fg = sql("SELECT value FROM fear_greed ORDER BY ts DESC LIMIT 1")["value"].iloc[0]
    votes["fng"] = 1 if fg < 20 else (-1 if fg > 80 else 0)
except: votes["fng"] = 0

for k, v in votes.items():
    print(f"  {k:<12} {v:+d}")

  trend        +1
  trend_lt     +1
  rsi          +0
  macd         -1
  drawdown     +0
  funding      +0
  fng          +0


## 3. Composite

In [4]:
total = sum(votes.values())
print("total votes:", total, " (range", -len(votes), "to", len(votes), ")")
print("bullish votes:", sum(1 for v in votes.values() if v > 0))
print("bearish votes:", sum(1 for v in votes.values() if v < 0))

total votes: 1  (range -7 to 7 )
bullish votes: 2
bearish votes: 1


## 4. Action

In [5]:
if total >= 3:   action, reason = "LONG",  f"strong bullish ({total} votes)"
elif total <= -3: action, reason = "SHORT", f"strong bearish ({total} votes)"
else:            action, reason = "FLAT",  f"mixed ({total} votes)"
action_block("MULTI-FACTOR", action, reason,
    price=float(last.close), conf=min(1.0, abs(total)/len(votes)),
    extra={"votes": total})

  MULTI-FACTOR
  time   : 2026-10-04 10:49 UTC
  price  : 2,687.23
  conf   : 0.14
  ACTION : FLAT
  REASON : mixed (1 votes)
  votes  : 1


{'action': 'FLAT',
 'reason': 'mixed (1 votes)',
 'price': 2687.23,
 'entry': None,
 'stop': None,
 'target': None,
 'conf': 0.14285714285714285}